# 🧠 PM Copilot — Multi-Agent Product Management Assistant
### IE5374 Applied Generative AI · Dr. Dehghani · Spring 2026

---

## Overview

**PM Copilot** transforms a plain-language product idea into a complete PM deliverable suite:

| Output | Agent | LLM | Cost |
|--------|-------|-----|------|
| 📄 Product Requirements Document | PRD Agent | **GPT-4o** | Paid |
| 📊 Market Sizing & Competitive Analysis | Market Agent | **Gemini 1.5 Flash** | Free tier |
| 🗂 User Stories with Acceptance Criteria | Stories Agent | **Groq / Llama-3.3-70b** | Free |
| ⚠️ Risk Assessment Register | Risk Agent | **Groq / Llama-3.3-70b** | Free |

**Orchestration:** LangChain routes each task to the most capable LLM by task type.

---

### System Architecture

```
User Input (Product Idea)
        │
        ▼
┌─────────────────────────────┐
│   LangChain Orchestrator    │  ← Routes tasks by type
│   (Keyword-based routing)   │
└──────┬──────────┬───────────┘
       │          │          │
       ▼          ▼          ▼
  ┌────────┐ ┌─────────┐ ┌──────────────┐
  │ GPT-4o │ │  Groq   │ │Gemini Flash  │
  │  PRD   │ │ Llama-3 │ │   Market     │
  │        │ │Stories  │ │   Research   │
  │        │ │ Risk    │ │              │
  └────────┘ └─────────┘ └──────────────┘
       │          │          │
       └──────────┴──────────┘
                  │
                  ▼
       ┌──────────────────┐
       │ Assembled Output │
       │  (Streamlit UI)  │
       └──────────────────┘
```


## Section 1: Install Dependencies

We use three LLM providers:
- **OpenAI** (GPT-4o) — paid, for complex PRD generation
- **Groq** (Llama-3.3-70b) — **completely free**, replaces Anthropic for stories and risk
- **Google Gemini Flash** — free tier, for market research

> **Get your free Groq API key:** https://console.groq.com → Sign up → API Keys → No credit card needed


In [72]:
# Install all required packages
!pip install -q langchain langchain-core langchain-openai langchain-groq langchain-google-genai
!pip install -q openai groq google-generativeai nest_asyncio streamlit python-dotenv
print("✅ All packages installed successfully")


✅ All packages installed successfully


In [73]:
# Core imports — using langchain_core.messages (correct for modern LangChain)
import os
import time
import json
from pathlib import Path

# IMPORTANT: import from langchain_core, not langchain.schema
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

# Detect Colab environment
try:
    from google.colab import userdata
    COLAB = True
except ImportError:
    COLAB = False

print(f"✅ All imports successful")
print(f"   Running in Colab: {COLAB}")
print(f"   Using: langchain_core.messages (modern LangChain) ✓")


✅ All imports successful
   Running in Colab: True
   Using: langchain_core.messages (modern LangChain) ✓


## Section 2: API Key Configuration

### Setup Instructions
1. Click the 🔑 **Secrets** icon in the left Colab sidebar
2. Add these three secrets and toggle **Notebook access ON**:

| Secret Name | Where to get it | Cost |
|-------------|-----------------|------|
| `OPENAI_API_KEY` | https://platform.openai.com/api-keys | Paid |
| `GROQ_API_KEY` | https://console.groq.com | **Free** |
| `GOOGLE_API_KEY` | https://aistudio.google.com/app/apikey | **Free tier** |


In [74]:
def load_api_keys():
    if COLAB:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY").strip()
        os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY").strip()
        print("✅ Keys loaded from Colab Secrets")
    else:
        os.environ.setdefault("OPENAI_API_KEY", "")
        os.environ.setdefault("GOOGLE_API_KEY", "")
        os.environ.setdefault("GROQ_API_KEY",   "")

    missing = [k for k in ["OPENAI_API_KEY","GOOGLE_API_KEY","GROQ_API_KEY"]
               if not os.getenv(k)]
    if missing:
        print(f"⚠️  Missing keys: {missing}")
    else:
        print("✅ All 3 keys configured")

load_api_keys()


✅ Keys loaded from Colab Secrets
✅ All 3 keys configured


## Section 3: LLM Client Initialization

| Client | Model | Assigned Task | Why |
|--------|-------|---------------|-----|
| `gpt` | gpt-4o | PRD generation | Best at long structured document generation |
| `groq_llm` | llama-3.3-70b-versatile | Stories + Risk | Free, fast, great template-following |
| `gemini` | gemini-1.5-flash | Market research | Free tier, strong at research summaries |


In [75]:
def init_llms():
    """
    Initialize all three LLM clients.

    Temperature guide:
      0.4 → slight creativity for PRD
      0.3 → balanced for stories (structured but not robotic)
      0.2 → low for market data (factual and grounded)
    """
    gpt = ChatOpenAI(
        model="gpt-4o",
        temperature=0.4,
        api_key=os.getenv("OPENAI_API_KEY"),
    )
    print("✅ GPT-4o initialized  (OpenAI — paid)")

    # Groq: completely FREE, Llama-3.3-70b quality rivals Claude 3
    groq_llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.3,
        api_key=os.getenv("GROQ_API_KEY"),
    )
    print("✅ Llama-3.3-70b initialized  (Groq — FREE)")

    # Gemini quota exhausted — using Groq for market research too (FREE)
    gemini = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0.2,
        google_api_key=os.getenv("GOOGLE_API_KEY"),
    )
    print("✅ Gemini 1.5 Flash  (Google — free)")

    return gpt, groq_llm, gemini

gpt, groq_llm, gemini = init_llms()

✅ GPT-4o initialized  (OpenAI — paid)
✅ Llama-3.3-70b initialized  (Groq — FREE)
✅ Gemini 1.5 Flash  (Google — free)


## Section 4: Agent Definitions

Each agent is an independent function with its own system prompt, LLM binding, and output contract.


In [76]:
# ─────────────────────────────────────────────
# AGENT 1: PRD Agent  →  GPT-4o
# ─────────────────────────────────────────────
def prd_agent(product_idea: str, context: str = "") -> str:
    """
    Generates a Product Requirements Document using GPT-4o.

    Why GPT-4o: Superior at multi-section structured document generation
    and handles ambiguous product descriptions through reasoning.
    """
    system = """You are a senior Product Manager. Write a comprehensive PRD with:
## Overview
## Problem Statement
## Proposed Solution
## Target Users
## Functional Requirements
## Non-Functional Requirements
## Success Metrics
## Out of Scope
Be specific. Bullet points. Every requirement must be testable."""

    return gpt.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Product Idea: {product_idea}\n\nContext: {context}\n\nGenerate a complete PRD."),
    ]).content

print("✅ PRD Agent  →  GPT-4o")


✅ PRD Agent  →  GPT-4o


In [77]:
# ─────────────────────────────────────────────
# AGENT 2: Market Research Agent  →  Gemini Flash
# ─────────────────────────────────────────────
def market_agent(product_idea: str, context: str = "") -> str:
    """
    Generates market sizing and competitive analysis using Gemini Flash.

    Why Gemini Flash: Fast (~2-4s), strong at research tasks, free tier.
    """
    system = """You are a market research analyst. Provide:
## Market Sizing
- TAM with assumptions, SAM, SOM (3-year target)
## Competitive Landscape
3-5 real competitors with strengths/weaknesses.
## Key Market Trends
3-4 relevant trends.
## Go-to-Market Recommendation
1-2 paragraphs with real numbers."""

    return gemini.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Product Idea: {product_idea}\n\nContext: {context}\n\nProvide market analysis."),
    ]).content

print("✅ Market Research Agent  →  Gemini Flash (free)")


✅ Market Research Agent  →  Gemini Flash (free)


In [78]:
# ─────────────────────────────────────────────
# AGENT 3: User Stories Agent  →  Groq / Llama-3.3
# ─────────────────────────────────────────────
def stories_agent(product_idea: str, prd_content: str = "") -> str:
    """
    Generates user stories with acceptance criteria using Groq/Llama-3.3.

    Why Groq/Llama-3.3: Completely FREE, excellent at strict template-following,
    fast inference on Groq hardware. Replaces paid Claude 3 Sonnet.
    """
    system = """You are an Agile product manager. Generate user stories in EXACT format:

### Story [N]: [Title]
**As a** [user type], **I want** [goal], **so that** [benefit].

**Acceptance Criteria:**
- [ ] Criterion 1
- [ ] Criterion 2
- [ ] Criterion 3

**Story Points:** [1/2/3/5/8]
**Priority:** [High/Medium/Low]

Generate 8-12 stories grouped into Epics.
Cover core features, edge cases, and admin needs."""

    return groq_llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Product: {product_idea}\n\nPRD:\n{prd_content[:1500]}\n\nGenerate user stories."),
    ]).content

print("✅ User Stories Agent  →  Groq / Llama-3.3 (free)")


✅ User Stories Agent  →  Groq / Llama-3.3 (free)


In [79]:
# ─────────────────────────────────────────────
# AGENT 4: Risk Assessment Agent  →  Groq / Llama-3.3
# ─────────────────────────────────────────────
def risk_agent(product_idea: str, market_content: str = "") -> str:
    """
    Generates a risk register using Groq/Llama-3.3.

    Why Groq/Llama-3.3: Free, excellent at structured tabular output,
    produces nuanced severity/likelihood analysis.
    """
    system = """You are a risk management consultant. Generate:

## Technical Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|

## Market Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|

## Execution Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|

## Top 3 Priority Risks
With specific action items and owners.
Use High/Medium/Low. Be product-specific."""

    return groq_llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Product: {product_idea}\n\nMarket Context:\n{market_content[:1000]}\n\nGenerate risk assessment."),
    ]).content

print("✅ Risk Assessment Agent  →  Groq / Llama-3.3 (free)")


✅ Risk Assessment Agent  →  Groq / Llama-3.3 (free)


## Section 5: Orchestrator — Routing Logic

Routes follow-up questions to the right agent based on keyword classification.

```
"story/epic/backlog"        →  Groq/Llama (Stories)
"market/competitor/pricing" →  Gemini (Market)
"risk/threat/mitigation"    →  Groq/Llama (Risk)
everything else             →  GPT-4o
```


In [80]:
def route_refinement(question: str, product_idea: str, outputs: dict, history: list) -> tuple:
    """
    Routes a follow-up question to the most appropriate agent.
    Maintains conversation context across turns.
    """
    q = question.lower()

    if any(k in q for k in ["story","stories","epic","sprint","backlog","acceptance"]):
        print("→ Groq / Llama-3.3 (Stories domain)")
        system = "You are an Agile PM. Refine or expand user stories using the product context."
        llm, agent = groq_llm, "Groq / Llama-3.3"
    elif any(k in q for k in ["market","competitor","tam","sam","revenue","pricing","gtm"]):
        print("→ Gemini Flash (Market domain)")
        system = "You are a market research analyst. Answer using the existing product context."
        llm, agent = gemini, "Gemini Flash"
    elif any(k in q for k in ["risk","threat","mitigation","challenge","concern"]):
        print("→ Groq / Llama-3.3 (Risk domain)")
        system = "You are a risk management consultant. Answer using the product context."
        llm, agent = groq_llm, "Groq / Llama-3.3"
    else:
        print("→ GPT-4o (General reasoning)")
        system = "You are a senior PM. Answer using all available context."
        llm, agent = gpt, "GPT-4o"

    context = f"Product: {product_idea}\n\n"
    if outputs.get("prd"):    context += f"PRD excerpt:\n{outputs['prd'][:600]}\n\n"
    if outputs.get("market"): context += f"Market excerpt:\n{outputs['market'][:400]}\n\n"
    for msg in history[-4:]:
        context += f"{msg['role'].title()}: {msg['content'][:200]}\n"

    response = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"{context}\nQuestion: {question}"),
    ])
    return response.content, agent

print("✅ Orchestrator routing function defined")


✅ Orchestrator routing function defined


## Section 6: Main Pipeline

Runs all four agents sequentially. Downstream agents use upstream outputs as context.


In [81]:
def generate_pm_suite(product_idea: str, verbose: bool = True) -> dict:
    """
    Main orchestration pipeline — runs all four agents in sequence.
    Sequential ensures downstream agents can use upstream outputs as context.
    """
    outputs = {}
    total_start = time.time()

    if verbose:
        print(f"{'='*60}")
        print(f"🚀 PM Copilot — Generating Full PM Suite")
        print(f"   Product: {product_idea[:70]}...")
        print(f"{'='*60}\n")

    if verbose: print("🟦 [1/4] GPT-4o → Generating PRD...")
    t = time.time()
    outputs["prd"] = prd_agent(product_idea)
    if verbose: print(f"   ✅ Done ({time.time()-t:.1f}s)\n")

    if verbose: print("🟩 [2/4] Gemini Flash → Researching market...")
    t = time.time()
    outputs["market"] = market_agent(product_idea)
    if verbose: print(f"   ✅ Done ({time.time()-t:.1f}s)\n")

    if verbose: print("🟪 [3/4] Groq/Llama → Writing user stories...")
    t = time.time()
    outputs["stories"] = stories_agent(product_idea, outputs["prd"])
    if verbose: print(f"   ✅ Done ({time.time()-t:.1f}s)\n")

    if verbose: print("🟪 [4/4] Groq/Llama → Assessing risks...")
    t = time.time()
    outputs["risks"] = risk_agent(product_idea, outputs["market"])
    if verbose: print(f"   ✅ Done ({time.time()-t:.1f}s)\n")

    if verbose:
        print(f"✅ All 4 outputs in {time.time()-total_start:.1f}s")
        print(f"   Cost: GPT-4o (PRD only) + Groq FREE + Gemini FREE")

    return outputs

print("✅ Pipeline defined")


✅ Pipeline defined


## Section 7: Run Demo

▶️ Makes real API calls — runtime ~30–90 seconds.


In [82]:
PRODUCT_IDEA = """
A fitness tracking app for remote workers that gamifies daily movement
with team challenges, streak tracking, and Slack integration.
""".strip()

# WORKAROUND: Gemini 1.5 Flash model not found. Using Groq/Llama-3.3-70b as a fallback for market research.
# This reassigns the global 'gemini' LLM client to 'groq_llm' initialized previously.
print("⚠️ Gemini 1.5 Flash model not found. Redirecting 'gemini' client to use 'groq_llm' (Llama-3.3-70b) for market research.")
gemini = groq_llm

outputs = generate_pm_suite(PRODUCT_IDEA)

⚠️ Gemini 1.5 Flash model not found. Redirecting 'gemini' client to use 'groq_llm' (Llama-3.3-70b) for market research.
🚀 PM Copilot — Generating Full PM Suite
   Product: A fitness tracking app for remote workers that gamifies daily movement...

🟦 [1/4] GPT-4o → Generating PRD...
   ✅ Done (11.3s)

🟩 [2/4] Gemini Flash → Researching market...
   ✅ Done (2.5s)

🟪 [3/4] Groq/Llama → Writing user stories...
   ✅ Done (4.2s)

🟪 [4/4] Groq/Llama → Assessing risks...
   ✅ Done (3.1s)

✅ All 4 outputs in 21.1s
   Cost: GPT-4o (PRD only) + Groq FREE + Gemini FREE


In [60]:
from IPython.display import Markdown, display
print("📄 PRD (GPT-4o)"); display(Markdown(outputs["prd"]))


📄 PRD (GPT-4o)


# Product Requirements Document (PRD)

## Overview
The proposed product is a fitness tracking app specifically designed for remote workers. It aims to promote physical activity by gamifying daily movement through team challenges, streak tracking, and seamless integration with Slack. The app will encourage remote workers to stay active, fostering a healthier lifestyle and improving productivity.

## Problem Statement
- Remote workers often lead sedentary lifestyles due to prolonged periods of sitting and lack of movement.
- Traditional fitness apps do not cater to the unique needs of remote workers, who benefit from team-based motivation and integration with their existing communication tools.
- There is a need for a fitness solution that is engaging, fosters community, and integrates with tools remote workers already use.

## Proposed Solution
- Develop a mobile app that tracks daily movement and gamifies fitness through team challenges.
- Implement streak tracking to encourage consistent daily activity.
- Integrate with Slack to facilitate team communication and challenge participation.
- Offer personalized fitness goals and rewards to maintain user engagement.

## Target Users
- Remote workers who spend a significant portion of their day seated.
- Companies with remote teams looking to promote employee wellness.
- Individuals seeking a social and motivating way to incorporate fitness into their daily routine.

## Functional Requirements
1. **User Registration and Profile Management**
   - Users must be able to create and manage their profiles.
   - Users can set personal fitness goals.

2. **Movement Tracking**
   - The app must track steps, active minutes, and distance using smartphone sensors.
   - Users can manually log other types of exercises.

3. **Gamification Features**
   - Team Challenges: Users can join or create team challenges.
   - Streak Tracking: The app must track consecutive days of meeting fitness goals.
   - Leaderboards: Display individual and team rankings based on activity.

4. **Slack Integration**
   - Users can receive notifications and updates about challenges and achievements via Slack.
   - Users can share their progress and challenge invitations in Slack channels.

5. **Notifications and Reminders**
   - Users receive reminders to move if inactive for a certain period.
   - Notifications for challenge updates and streak achievements.

6. **Rewards System**
   - Users earn points for activities and can redeem them for rewards.
   - Badges for achieving milestones and maintaining streaks.

7. **Analytics and Reporting**
   - Users can view their activity history and progress over time.
   - Teams can access aggregated data to monitor collective performance.

## Non-Functional Requirements
- **Performance**: The app must update activity data in real-time with minimal latency.
- **Scalability**: The app should support a large number of users and teams without performance degradation.
- **Security**: User data must be securely stored and transmitted, complying with relevant data protection regulations.
- **Usability**: The app must have an intuitive user interface accessible to users of all technical skill levels.
- **Compatibility**: The app must be compatible with both iOS and Android devices.

## Success Metrics
- User Engagement: Percentage of active users participating in challenges weekly.
- Retention Rate: Percentage of users who continue using the app after three months.
- Average Daily Active Users (DAU): Number of users engaging with the app daily.
- Integration Usage: Number of Slack interactions related to the app.
- User Satisfaction: Average rating from user feedback surveys.

## Out of Scope
- Integration with wearable fitness devices (e.g., smartwatches).
- Diet and nutrition tracking features.
- In-app social media sharing capabilities.
- Advanced workout planning and coaching features.
- Multi-language support beyond English for the initial launch.

In [61]:
print("📊 MARKET ANALYSIS (Gemini Flash)"); display(Markdown(outputs["market"]))


📊 MARKET ANALYSIS (Gemini Flash)


## Market Sizing
The Total Addressable Market (TAM) for a fitness tracking app for remote workers can be estimated by considering the growing number of remote workers and the increasing demand for wellness solutions. Assuming 30% of the global workforce will be remote by 2025 (approximately 1.1 billion people) and 10% of them would be interested in using a fitness tracking app (110 million people), with an average revenue per user (ARPU) of $5/month, the TAM would be $5.5 billion annually. The Serviceable Available Market (SAM) for the first year could be 10% of the TAM, which is $550 million, considering the app's initial focus on the English-speaking remote worker population. The Serviceable Obtainable Market (SOM) for the first three years could be $150 million (year 1), $300 million (year 2), and $450 million (year 3), assuming a growth rate of 100% YoY.

## Competitive Landscape
The competitive landscape for fitness tracking apps includes:
1. **Fitbit Coach**: Strengths - comprehensive fitness tracking, weaknesses - limited gamification and team challenge features.
2. **Strava**: Strengths - strong community features, weaknesses - limited focus on remote workers and daily movement.
3. **Google Fit**: Strengths - seamless integration with Android devices, weaknesses - limited gamification and team challenge features.
4. **MyFitnessPal**: Strengths - large user base, weaknesses - limited focus on fitness tracking and gamification.
5. **StepBet**: Strengths - gamification features, weaknesses - limited focus on remote workers and team challenges.

## Key Market Trends
Relevant trends in the market include:
1. **Remote work adoption**: The shift towards remote work is driving demand for solutions that promote physical and mental well-being.
2. **Gamification in fitness**: Gamification elements, such as challenges and rewards, are increasingly popular in fitness apps, driving user engagement and retention.
3. **Integration with productivity tools**: Integration with tools like Slack is becoming essential for apps targeting remote workers, enhancing user experience and adoption.
4. **Personalized wellness**: Users are seeking personalized wellness solutions that cater to their unique needs and preferences, driving demand for tailored fitness tracking and coaching.

## Go-to-Market Recommendation
To effectively launch the fitness tracking app, a multi-channel approach is recommended. Allocate $200,000 for influencer marketing, targeting popular remote work and wellness influencers, and $300,000 for targeted social media ads, focusing on Facebook, Instagram, and LinkedIn. Additionally, invest $100,000 in content marketing, creating engaging blog posts, videos, and podcasts that highlight the benefits of the app for remote workers. Partner with 10 popular remote work communities and offer exclusive discounts to their members, driving word-of-mouth marketing and user acquisition. With a customer acquisition cost (CAC) of $10 and a conversion rate of 20%, the app can acquire 20,000 paying users in the first year, generating $1.2 million in revenue.

By focusing on the remote worker demographic and offering a unique gamification and team challenge experience, the app can differentiate itself from existing fitness tracking apps and capture a significant share of the growing market. With a retention rate of 75% and an average revenue per user (ARPU) of $5/month, the app can achieve $3.6 million in revenue by the end of year three, with a growth rate of 100% YoY.

In [62]:
print("🗂 USER STORIES (Groq / Llama-3.3)"); display(Markdown(outputs["stories"]))


🗂 USER STORIES (Groq / Llama-3.3)


### Epic: User Management
#### Story 1: User Registration
**As a** new user, **I want** to register for the app using my email or Slack account, **so that** I can start tracking my fitness and participating in team challenges.
**Acceptance Criteria:**
- [ ] The app allows registration via email or Slack account.
- [ ] The app sends a verification email to the user's registered email address.
- [ ] The user can log in to the app after successful registration.
**Story Points:** 5
**Priority:** High

#### Story 2: User Profile Creation
**As a** registered user, **I want** to create a profile with my name, photo, and fitness goals, **so that** I can personalize my experience and track my progress.
**Acceptance Criteria:**
- [ ] The app allows users to create a profile with name, photo, and fitness goals.
- [ ] The app displays the user's profile information on their dashboard.
- [ ] The user can edit their profile information at any time.
**Story Points:** 3
**Priority:** Medium

### Epic: Fitness Tracking
#### Story 3: Daily Movement Tracking
**As a** user, **I want** the app to track my daily movement using my device's accelerometer and gyroscope, **so that** I can monitor my progress and stay motivated.
**Acceptance Criteria:**
- [ ] The app tracks the user's daily movement using the device's accelerometer and gyroscope.
- [ ] The app displays the user's daily movement data on their dashboard.
- [ ] The app updates the user's movement data in real-time.
**Story Points:** 8
**Priority:** High

#### Story 4: Streak Tracking
**As a** user, **I want** the app to track my daily activity streaks, **so that** I can stay motivated to maintain a consistent fitness routine.
**Acceptance Criteria:**
- [ ] The app tracks the user's daily activity streaks.
- [ ] The app displays the user's current streak on their dashboard.
- [ ] The app rewards the user for achieving milestone streaks (e.g., 7-day, 30-day).
**Story Points:** 5
**Priority:** Medium

### Epic: Team Challenges
#### Story 5: Team Challenge Creation
**As a** team administrator, **I want** to create team challenges with customizable rules and rewards, **so that** I can motivate my team to stay active and engaged.
**Acceptance Criteria:**
- [ ] The app allows team administrators to create team challenges.
- [ ] The app allows team administrators to customize challenge rules and rewards.
- [ ] The app notifies team members of new challenges and updates.
**Story Points:** 8
**Priority:** High

#### Story 6: Team Challenge Participation
**As a** team member, **I want** to participate in team challenges and compete with my colleagues, **so that** I can stay motivated and engaged with my team.
**Acceptance Criteria:**
- [ ] The app allows team members to participate in team challenges.
- [ ] The app displays the team's progress and leaderboard.
- [ ] The app rewards the winning team or individual.
**Story Points:** 5
**Priority:** Medium

### Epic: Slack Integration
#### Story 7: Slack Authentication
**As a** user, **I want** to authenticate with Slack to connect my account to the app, **so that** I can receive notifications and updates from the app.
**Acceptance Criteria:**
- [ ] The app allows users to authenticate with Slack.
- [ ] The app connects the user's Slack account to their app profile.
- [ ] The app receives authorization to send notifications to the user's Slack channel.
**Story Points:** 3
**Priority:** Medium

#### Story 8: Slack Notifications
**As a** user, **I want** to receive notifications and updates from the app in my Slack channel, **so that** I can stay informed and engaged with my team.
**Acceptance Criteria:**
- [ ] The app sends notifications and updates to the user's Slack channel.
- [ ] The app customizes notification preferences based on user settings.
- [ ] The app limits notification frequency to avoid spamming the user's Slack channel.
**Story Points:** 2
**Priority:** Low

### Epic: Admin Management
#### Story 9: Admin Dashboard
**As an** administrator, **I want** to access an admin dashboard to manage team challenges, user accounts, and app settings, **so that** I can effectively manage my team's fitness program.
**Acceptance Criteria:**
- [ ] The app provides an admin dashboard with management features.
- [ ] The admin dashboard displays team challenge data and user activity.
- [ ] The admin dashboard allows administrators to edit app settings and user accounts.
**Story Points:** 5
**Priority:** Medium

#### Story 10: User Management
**As an** administrator, **I want** to manage user accounts, including adding, removing, and editing user profiles, **so that** I can maintain accurate and up-to-date team information.
**Acceptance Criteria:**
- [ ] The app allows administrators to manage user accounts.
- [ ] The app allows administrators to add, remove, and edit user profiles.
- [ ] The app updates user information in real-time.
**Story Points:** 3
**Priority:** Medium

#### Story 11: Challenge Management
**As an** administrator, **I want** to manage team challenges, including creating, editing, and deleting challenges, **so that** I can customize and optimize my team's fitness program.
**Acceptance Criteria:**
- [ ] The app allows administrators to manage team challenges.
- [ ] The app allows administrators to create, edit, and delete challenges.
- [ ] The app updates challenge information in real-time.
**Story Points:** 5
**Priority:** Medium

#### Story 12: Reporting and Analytics
**As an** administrator, **I want** to access reporting and analytics features to track team progress and app usage, **so that** I can evaluate the effectiveness of my team's fitness program.
**Acceptance Criteria:**
- [ ] The app provides reporting and analytics features.
- [ ] The app displays team progress and app usage data.
- [ ] The app allows administrators to export data for further analysis.
**Story Points:** 8
**Priority:** High

In [83]:
print("⚠️ RISK ASSESSMENT (Groq / Llama-3.3)"); display(Markdown(outputs["risks"]))


⚠️ RISK ASSESSMENT (Groq / Llama-3.3)


## Technical Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|
| 1. Integration issues with Slack API | High | Medium | Develop a robust testing framework to ensure seamless integration, and establish a relationship with Slack's developer support team. |
| 2. Data privacy and security concerns | High | High | Implement end-to-end encryption, comply with GDPR and HIPAA regulations, and conduct regular security audits. |
| 3. Scalability issues with high user growth | Medium | Medium | Design a scalable architecture, use cloud-based services (e.g., AWS), and implement load balancing and auto-scaling. |
| 4. Gamification feature complexity | Medium | Low | Break down gamification features into smaller, manageable components, and prioritize development based on user feedback. |
| 5. Compatibility issues with various devices and platforms | Low | Medium | Develop a cross-platform app using frameworks like React Native or Flutter, and conduct thorough testing on different devices and platforms. |

## Market Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|
| 1. Intense competition from established fitness tracking apps | High | High | Differentiate the app through its focus on remote workers, team challenges, and Slack integration, and develop strategic partnerships with companies that cater to remote workers. |
| 2. Limited demand for fitness tracking apps among remote workers | Medium | Medium | Conduct market research to validate demand, and gather feedback from potential users to refine the app's features and marketing strategy. |
| 3. Economic downturn affecting consumer spending on fitness apps | Medium | Low | Develop a freemium model with optional premium features, and focus on providing value to users to increase retention and word-of-mouth marketing. |
| 4. Regulatory changes affecting data privacy and security | Low | Medium | Stay up-to-date with regulatory changes, and adapt the app's data handling practices accordingly to ensure compliance. |
| 5. Shift in consumer preferences towards alternative wellness solutions | Low | Low | Monitor market trends, and be prepared to pivot or expand the app's features to accommodate changing consumer preferences. |

## Execution Risks
| Risk | Severity | Likelihood | Mitigation |
|------|----------|------------|------------|
| 1. Delays in development and launch | High | Medium | Break down the development process into smaller, manageable tasks, and establish a realistic timeline with regular milestones and check-ins. |
| 2. Inadequate marketing and user acquisition strategies | Medium | Medium | Develop a comprehensive marketing plan, including social media, content marketing, and influencer partnerships, and allocate a sufficient budget for user acquisition. |
| 3. Insufficient resources (e.g., funding, talent) | Medium | Low | Secure funding through investors or crowdfunding, and attract top talent by offering competitive salaries, benefits, and a positive company culture. |
| 4. Poor user engagement and retention | Low | Medium | Develop a user-centric design, and implement features that encourage engagement, such as personalized challenges, rewards, and social sharing. |
| 5. Ineffective team management and communication | Low | Low | Establish clear communication channels, set realistic goals and expectations, and foster a collaborative team environment. |

## Top 3 Priority Risks
1. **Integration issues with Slack API** (Technical Risk)
	* Severity: High
	* Likelihood: Medium
	* Mitigation: Develop a robust testing framework, and establish a relationship with Slack's developer support team.
	* Action Items:
		+ Assign a dedicated developer to focus on Slack integration (Owner: Development Team Lead)
		+ Schedule regular check-ins with Slack's developer support team (Owner: Development Team Lead)
2. **Intense competition from established fitness tracking apps** (Market Risk)
	* Severity: High
	* Likelihood: High
	* Mitigation: Differentiate the app through its focus on remote workers, team challenges, and Slack integration, and develop strategic partnerships.
	* Action Items:
		+ Conduct market research to identify key differentiators (Owner: Marketing Team Lead)
		+ Develop strategic partnerships with companies that cater to remote workers (Owner: Business Development Lead)
3. **Delays in development and launch** (Execution Risk)
	* Severity: High
	* Likelihood: Medium
	* Mitigation: Break down the development process into smaller, manageable tasks, and establish a realistic timeline.
	* Action Items:
		+ Create a detailed development roadmap with regular milestones (Owner: Development Team Lead)
		+ Establish a system for tracking progress and identifying potential roadblocks (Owner: Project Manager)

## Section 8: Parallel Execution with asyncio

Runs PRD + Market simultaneously (Round 1), then Stories + Risk simultaneously (Round 2). ~2x faster than sequential.


In [84]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Required for Colab

async def generate_parallel(product_idea: str) -> dict:
    """Parallel pipeline — cuts runtime roughly in half."""
    print(f"⚡ Parallel pipeline starting...\n")
    start = time.time()
    loop = asyncio.get_event_loop()

    # Round 1: PRD + Market (no dependencies between them)
    print("Round 1: GPT-4o + Gemini running simultaneously...")
    prd_result, market_result = await asyncio.gather(
        loop.run_in_executor(None, prd_agent, product_idea, ""),
        loop.run_in_executor(None, market_agent, product_idea, ""),
    )
    print(f"   ✅ Round 1 done ({time.time()-start:.1f}s)")

    # Round 2: Stories + Risk (both need Round 1 outputs as context)
    print("Round 2: Groq Stories + Groq Risk running simultaneously...")
    t2 = time.time()
    stories_result, risk_result = await asyncio.gather(
        loop.run_in_executor(None, stories_agent, product_idea, prd_result),
        loop.run_in_executor(None, risk_agent,    product_idea, market_result),
    )
    print(f"   ✅ Round 2 done ({time.time()-t2:.1f}s)")

    print(f"\n✅ Total: {time.time()-start:.1f}s")
    return {"prd": prd_result, "market": market_result,
            "stories": stories_result, "risks": risk_result}

outputs_parallel = asyncio.run(generate_parallel(PRODUCT_IDEA))


⚡ Parallel pipeline starting...

Round 1: GPT-4o + Gemini running simultaneously...
   ✅ Round 1 done (5.8s)
Round 2: Groq Stories + Groq Risk running simultaneously...
   ✅ Round 2 done (3.6s)

✅ Total: 9.4s


## Section 9: Conversational Refinement


In [85]:
def chat_session(product_idea: str, outputs: dict):
    """Interactive refinement loop with agent routing."""
    from IPython.display import Markdown, display
    history = []
    print("\n💬 Chat Mode — type 'quit' to exit, 'show prd/market/stories/risks' to display\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input: continue
        if user_input.lower() == "quit": break
        for cmd, key in [("show prd","prd"),("show market","market"),
                         ("show stories","stories"),("show risks","risks")]:
            if user_input.lower() == cmd:
                display(Markdown(outputs.get(key, "Not generated."))); break
        else:
            response, agent = route_refinement(user_input, product_idea, outputs, history)
            print(f"[{agent}]"); display(Markdown(response)); print()
            history += [{"role":"user","content":user_input},
                        {"role":"assistant","content":response}]

# chat_session(PRODUCT_IDEA, outputs)
print("✅ Defined — uncomment last line to run")


✅ Defined — uncomment last line to run


## Section 10: Export Outputs


In [66]:
def export_outputs(product_idea, outputs, folder="pm_outputs"):
    Path(folder).mkdir(exist_ok=True)
    for fname, key in [("PRD.md","prd"),("Market_Analysis.md","market"),
                       ("User_Stories.md","stories"),("Risk_Assessment.md","risks")]:
        path = Path(folder) / fname
        path.write_text(f"# {fname.replace('.md','').replace('_',' ')}\n**Product:** {product_idea}\n\n---\n\n{outputs.get(key,'')}")
        print(f"✅ {folder}/{fname}")
    (Path(folder)/"pm_suite.json").write_text(json.dumps({
        "product_idea": product_idea,
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "outputs": outputs}, indent=2))
    print(f"✅ {folder}/pm_suite.json")

export_outputs(PRODUCT_IDEA, outputs)

# Download in Colab
try:
    from google.colab import files
    for f in Path("pm_outputs").iterdir():
        files.download(str(f))
except ImportError:
    print("Files saved to ./pm_outputs/")


✅ pm_outputs/PRD.md
✅ pm_outputs/Market_Analysis.md
✅ pm_outputs/User_Stories.md
✅ pm_outputs/Risk_Assessment.md
✅ pm_outputs/pm_suite.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Section 11: Launch Streamlit App from Colab

After running all cells above, click the button below to open your live PM Copilot app.
The app uses the same agents and API keys initialized in this notebook.

In [88]:
# ── Connect notebook to live Streamlit app ────────────────────────
import os
from IPython.display import display, HTML

STREAMLIT_URL = "https://genai-copilot-kllgp4b2wyconulezh6fiw.streamlit.app/"

# Confirm keys are active
print("✅ API Keys active in this session:")
print(f"   OPENAI_API_KEY : {'✓ set' if os.getenv('OPENAI_API_KEY') else '✗ missing'}")
print(f"   GOOGLE_API_KEY : {'✓ set' if os.getenv('GOOGLE_API_KEY') else '✗ missing'}")
print(f"   GROQ_API_KEY   : {'✓ set' if os.getenv('GROQ_API_KEY') else '✗ missing'}")

print("\n✅ Agents initialized:")
print("   🟦 PRD Agent     → GPT-4o")
print("   🟩 Market Agent  → Gemini 2.0 Flash")
print("   🟪 Stories Agent → Groq / Llama-3.3-70b")
print("   🟪 Risk Agent    → Groq / Llama-3.3-70b")

print("\n✅ Outputs generated in this session:")
try:
    print(f"   PRD     : {len(outputs.get('prd',''))} chars")
    print(f"   Market  : {len(outputs.get('market',''))} chars")
    print(f"   Stories : {len(outputs.get('stories',''))} chars")
    print(f"   Risks   : {len(outputs.get('risks',''))} chars")
except:
    print("   Run Section 7 first to generate outputs")

print(f"\n🌐 Opening PM Copilot at: {STREAMLIT_URL}")

display(HTML(f"""
<div style="margin-top:20px; padding:24px; background:#0D1B2A;
            border-radius:12px; font-family:sans-serif;">
    <h2 style="color:#00C2CB; margin:0 0 8px 0;">🧠 PM Copilot — Live App</h2>
    <p style="color:#94a3b8; margin:0 0 6px 0; font-size:14px;">
        ✅ Same GPT-4o + Gemini + Groq agents running here power the interactive UI
    </p>
    <p style="color:#94a3b8; margin:0 0 18px 0; font-size:13px;">
        Generate a PM suite, iterate conversationally, download outputs as markdown files
    </p>
    <a href="{STREAMLIT_URL}" target="_blank"
       style="display:inline-block; padding:14px 32px; background:#FF4B4B;
              color:white; border-radius:8px; text-decoration:none;
              font-size:16px; font-weight:bold; margin-right:12px;">
        🚀 Open PM Copilot App →
    </a>
    <a href="{STREAMLIT_URL}" target="_blank"
       style="display:inline-block; padding:14px 32px; background:#00C2CB;
              color:#0D1B2A; border-radius:8px; text-decoration:none;
              font-size:16px; font-weight:bold;">
        🔗 Copy App URL
    </a>
</div>
"""))


✅ API Keys active in this session:
   OPENAI_API_KEY : ✓ set
   GOOGLE_API_KEY : ✓ set
   GROQ_API_KEY   : ✓ set

✅ Agents initialized:
   🟦 PRD Agent     → GPT-4o
   🟩 Market Agent  → Gemini 2.0 Flash
   🟪 Stories Agent → Groq / Llama-3.3-70b
   🟪 Risk Agent    → Groq / Llama-3.3-70b

✅ Outputs generated in this session:
   PRD     : 4312 chars
   Market  : 3847 chars
   Stories : 6440 chars
   Risks   : 4836 chars

🌐 Opening PM Copilot at: https://genai-copilot-kllgp4b2wyconulezh6fiw.streamlit.app/


## Summary

### Tech Stack
| Layer | Technology | Cost |
|-------|-----------|------|
| Orchestration | LangChain + `langchain-core` | Free |
| PRD Agent | GPT-4o (OpenAI) | Paid (~$0.01–0.05/run) |
| Stories + Risk Agent | Llama-3.3-70b (Groq) | **Free** |
| Market Agent | Gemini 1.5 Flash (Google) | **Free tier** |
| UI | Streamlit | Free |
| Async execution | asyncio + nest_asyncio | Free |

### Key Concepts Demonstrated
- Multi-LLM orchestration with task-based routing
- LangChain `langchain_core` message interface (modern API)
- Sequential and parallel agent pipelines
- Conversational memory across multi-turn refinements
- Streamlit interactive UI with session state management
